# CRISPRtracrRNA Prediction

This notebook demonstrates `run_crispr_tracr`, which scans nucleotide sequences for tracrRNA candidates using the CRISPRtracrRNA tool from the Backofen Lab. tracrRNA is required by type II CRISPR-Cas9 systems to form the guide RNA complex; identifying it is a critical step when characterizing novel Cas9 orthologs or validating candidate loci in a CRISPR engineering pipeline.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("crispr_tracr")
display_overview("crispr_tracr")
display_docs_section("crispr_tracr", "Background")

# CRISPRtracrRNA

> CRISPRtracrRNA predicts [tracrRNA](https://en.wikipedia.org/wiki/Trans-activating_crRNA) (trans-activating CRISPR RNA) sequences from nucleotide [CRISPR](https://en.wikipedia.org/wiki/CRISPR) loci using [covariance models](http://eddylab.org/infernal/) and RNA-RNA interaction prediction. It identifies tracrRNA candidates, scores them by E-value and anti-repeat similarity, and predicts the anti-repeat interaction structure via [IntaRNA](https://rna.informatik.uni-freiburg.de/IntaRNA/).

## Background

**What does this tool measure/predict?**
This tool predicts the location and quality of tracrRNA sequences within CRISPR loci. The tracrRNA is a non-coding RNA essential for [Type II CRISPR-Cas](https://en.wikipedia.org/wiki/CRISPR#Class_2) systems; it base-pairs with the CRISPR repeat (via its anti-repeat region) and forms a complex with [Cas9](https://en.wikipedia.org/wiki/Cas9) to guide DNA cleavage.

**Why is this important?**
For a CRISPR-Cas9 system to function, three components must be present: the Cas9 protein, the [crRNA](https://en.wikipedia.org/wiki/CRISPR#crRNA) (from the CRISPR array), and the tracrRNA. Detecting the tracrRNA is essential for:

* Confirming that a CRISPR locus encodes a complete, functional Type II system
* Designing [single-guide RNAs](https://en.wikipedia.org/wiki/Guide_RNA) (sgRNAs) by fusing crRNA and tracrRNA sequences
* Validating generated CRISPR systems (e.g., from Evo1) for functional completeness
* Discovering novel CRISPR-Cas9 systems in metagenomic data

**Scientific foundation:**
CRISPRtracrRNA uses [Infernal](http://eddylab.org/infernal/) covariance models (CMs) to detect tracrRNA candidates. Covariance models are probabilistic models of RNA sequence and secondary structure; similar to [profile HMMs](https://en.wikipedia.org/wiki/Hidden_Markov_model) but extended to capture base-pairing (covariance). After identifying tracrRNA candidates, the tool uses [IntaRNA](https://rna.informatik.uni-freiburg.de/IntaRNA/) to predict the RNA-RNA interaction between the tracrRNA's anti-repeat region and the CRISPR repeat, providing a structural validation of the prediction.

## Available tools

In [2]:
display_available_tools("crispr_tracr")

- **`run_crispr_tracr()`** — Predict tracrRNA sequences from nucleotide CRISPR loci

### `run_crispr_tracr`

`run_crispr_tracr` predicts tracrRNA sequences from nucleotide sequences that contain CRISPR loci. The tool evaluates anti-repeat complementarity and uses IntaRNA to compute RNA-RNA interaction energies, returning predicted tracrRNA coordinates and interaction energy scores for each input sequence. In practice the input should be a confirmed CRISPR-containing genomic region, typically identified by a tool like MinCED in an earlier analysis step.

In [3]:
from proto_tools import (
    CrisprTracrInput,
    CrisprTracrConfig,
    run_crispr_tracr,
)

In [4]:
# Display input docs
display_api_reference("crispr_tracr", "input", "run_crispr_tracr")

# Provide a genomic sequence containing a CRISPR locus
# In practice, use a real CRISPR-containing genomic region
inputs = CrisprTracrInput(
    sequences=["ATGCGTAAACGATTGCAGT" * 500],
    sequence_ids=["locus_1"],
)

**Input** — `CrisprTracrInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Nucleotide sequence(s) to predict tracrRNA from. Each sequence should contain a CRISPR locus. |
| `sequence_ids` | `array` |  | Optional sequence identifiers. |

In [5]:
# Display config docs
display_api_reference("crispr_tracr", "config", "run_crispr_tracr")

# Type II model for Cas9-associated tracrRNA prediction | see docs above for all fields
config = CrisprTracrConfig(model_type="II")

**Config** — `CrisprTracrConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_type` | `enum` | `II` | Type of CRISPR model to use. Available options: `II`, `all` |
| `run_type` | `enum` | `complete_run` | Pipeline mode (complete\_run or model\_only). Available options: `complete_run`, `model_only` |
| `num_workers` | `integer` |  | Number of parallel workers. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the tracrRNA prediction
result = run_crispr_tracr(inputs, config)

Removing environment at /home/bviggiano/codebases/proto-bio/proto_tool_envs/crispr_tracr_env
Setting up environment for crispr_tracr
ERROR: Environment setup failed for crispr_tracr (exit 1):
Cloning into '/home/bviggiano/codebases/proto-bio/proto_tool_envs/crispr_tracr_env/CRISPRtracrRNA'...
Cloning CRISPRidentify into CRISPRtracrRNA tools directory...
Cloning into '/home/bviggiano/codebases/proto-bio/proto_tool_envs/crispr_tracr_env/CRISPRtracrRNA/tools/CRISPRidentify/CRISPRidentify'...
Cloning CRISPRcasIdentifier into CRISPRtracrRNA tools directory...
Cloning into '/home/bviggiano/codebases/proto-bio/proto_tool_envs/crispr_tracr_env/CRISPRtracrRNA/tools/CRISPRcasIdentifier/CRISPRcasIdentifier'...
Creating isolated conda environment (Python 3.8 + scikit-learn 0.22)...
CRISPRidentify's pickled models require sklearn 0.22 (incompatible with 3.12).
Using /home/bviggiano/codebases/proto-bio/proto_tool_envs/crispr_tracr_env/conda_deps to avoid polluting base env...
ERROR: CRISPRtracrRNA r

In [7]:
# Display output docs
display_api_reference("crispr_tracr", "output", "run_crispr_tracr")

# Report predicted tracrRNA coordinates and interaction energy per input sequence
for pred in result.predictions:
    if pred.has_tracr:
        print(f"{pred.sequence_id}: tracrRNA at {pred.tracr_start}-{pred.tracr_end}")
        print(f"  Interaction energy: {pred.interaction_energy} kcal/mol")
    else:
        print(f"{pred.sequence_id}: no tracrRNA detected")

**Output** — `CrisprTracrOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `predictions` | `List[TracrPrediction]` |  | Per-sequence tracrRNA predictions. |
| `sequence_id` | `string` | required | ID of the input sequence |
| `tracr_start` | `integer` |  | Start position of predicted tracrRNA |
| `tracr_end` | `integer` |  | End position of predicted tracrRNA |
| `tracr_hit` | `string` |  | tracrRNA hit description |
| `interaction_energy` | `number` |  | IntaRNA interaction energy in kcal/mol, more negative = stronger (complete\_run mode) |
| `anti_repeat_similarity_coverage_multiplication` | `number` |  | Anti-repeat similarity x coverage score |
| `intarna_anti_repeat_interaction` | `string` |  | IntaRNA anti-repeat interaction prediction |

## Export Results

TracrRNA predictions can be exported to CSV or JSON. CSV produces one row per input sequence with predicted coordinates, interaction energy, and anti-repeat scores. JSON preserves the same fields for programmatic consumption.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="crispr_tracr_results", export_path=output_dir, file_format="json")
print(f"tracrRNA predictions exported to {output_dir}")

tracrRNA predictions exported to example_output
